## Config laden & Init

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import yaml
import optuna
import shap
import warnings

from sklearn.linear_model import RidgeCV, LinearRegression, LassoCV
from sklearn.exceptions import ConvergenceWarning
from statsmodels.tsa.statespace.sarimax import SARIMAX
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# Load Constants
NB_PATH = Path(globals()["__vsc_ipynb_file__"]).resolve()
ROOT = NB_PATH.parent.parent 
FIG_PATH = ROOT / "reports" / "figures"
CONFIG_PATH = ROOT / "configs" / "config.yaml"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

CSV_PATH = ROOT / config["data"]["csv_path"]

# Initialisierung
rs = config["modeling"]["random_state"]
np.random.seed(rs)
dpi = config["modeling"]["dpi"]
df = pd.read_csv(CSV_PATH, sep=config["data"]["sep"])
figsize = config["modeling"]["figsize"]
blue_color = config["modeling"]["blue_color"]
grey_color = config["modeling"]["grey_color"]
default_color = config["modeling"]["default_color"]
initial_train_size = config["modeling"]["train_size"]
optuna_trials = config["modeling"]["optuna_trials"]
start_block = config["modeling"]["start_block"]
patience_high = config["modeling"]["patience_high"]
patience_low = config["modeling"]["patience_low"]

# CSV laden
df.columns = df.columns.str.strip()
df["KW"] = df.index
df["Datum"] = pd.to_datetime(config["data"]["start_date"]) + pd.to_timedelta(df["KW"] * 7, unit="D")
df["Monat_Jahr"] = df["Datum"].dt.to_period("M").dt.to_timestamp()

# Featureset
features = config["featuresset"]["features"]
target = config["featuresset"]["target"]
stakeholder_features = config["stakeholder_features"]
optional_features = config["optional_features"]


# Zielvariable
df[target] = df["Alle Abschlüsse"]

# Monatsaggregation
df_monthly = df.groupby(config["aggregation"]["groupby_col"]).agg(config["aggregation"]["agg_map"])

X = df_monthly[features].dropna()
y = df_monthly[target].loc[X.index]

for f in stakeholder_features:
    df_monthly[f"{f} (Vormonat)"] = df_monthly[f].shift(1)

df_monthly["Zielvariable Gesamt (Vormonat)"] = df_monthly[target].shift(1)

## Optuna optimiert - Lineare Regression mit Expanding Window

In [ ]:
# Final definierte Features
stakeholder_features = [f"{f} (Vormonat)" for f in stakeholder_features]

optional_features = optional_features + ["Zielvariable Gesamt (Vormonat)"]
y_target = df_monthly["Zielvariable_Gesamt"]

# Forecast-Funktion
def run_forecast(X_raw, y, start_block=start_block):
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), index=X_raw.index, columns=X_raw.columns)
    trues, preds, dates = [], [], []
    for i in range(start_block, len(X_scaled)):
        X_train, y_train = X_scaled.iloc[:i], y.iloc[:i]
        X_test, y_test = X_scaled.iloc[i:i+1], y.iloc[i:i+1]
        model = LinearRegression(positive=True)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)[0]
        preds.append(y_pred)
        trues.append(y_test.values[0])
        dates.append(y_test.index[0])
    r2 = r2_score(trues, preds)
    mae = mean_absolute_error(trues, preds)
    err = mae / np.mean(trues) * 100
    return dates, trues, preds, r2, mae, err

# Early Stopping Callback
def early_stopping_callback(patience=patience_high):
    best_value = [float("inf")]
    counter = [0]
    def callback(study, trial):
        if trial.value < best_value[0]:
            best_value[0] = trial.value
            counter[0] = 0
        else:
            counter[0] += 1
        if counter[0] >= patience:
            raise optuna.exceptions.TrialPruned()
    return callback

# Optuna-Funktion
def optimize_features(fixed_features, optional_features, y, start_block=start_block, n_trials=optuna_trials, patience=patience_high):
    study = optuna.create_study(direction="minimize")
    try:
        def objective(trial):
            selected = [f for f in optional_features if trial.suggest_categorical(f, [True, False])]
            selected += fixed_features
            if len(selected) == 0:
                return float("inf")
            X = df_monthly[selected].dropna()
            y_clean = y.loc[X.index]
            _, _, _, _, mae, _ = run_forecast(X, y_clean, start_block)
            return mae
        study.optimize(objective, n_trials=n_trials, callbacks=[early_stopping_callback(patience)])
    except optuna.exceptions.TrialPruned:
        pass
    best_features = [f for f in optional_features if study.best_trial.params.get(f, False)] + fixed_features
    return best_features, study.best_value

# Lineare Regression – Optuna-optimiert
features_final, _ = optimize_features(stakeholder_features, optional_features, y_target)
X_final = df_monthly[features_final].dropna()
dates_final, trues_final, preds_final, r2_final, mae_final, err_final = run_forecast(X_final, y_target.loc[X_final.index])

# Plot-Funktion
def plot_forecast(dates, trues, preds, r2, mae, err_rate, features, title_suffix):
    # Forecast-Plot
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    full_y = df_monthly["Zielvariable_Gesamt"]
    ax.plot(full_y.index, full_y, label="Reale Abschlüsse", color=grey_color, marker="o")
    ax.plot(dates, preds, label=f"{title_suffix}", color=blue_color, linestyle="--", marker="x")
    ax.axvline(x=dates[0], color=grey_color, linestyle="--", label="Beginn Prognose")
    ax.set_title(f"{title_suffix}\nR² = {r2:.3f}, MAE = {mae:.2f}, Fehlerquote = {err_rate:.2f}%", fontsize=13)
    features_string = ", ".join(features)
    plt.suptitle(f"Verwendete Features:\n{features_string}", y=1.05, fontsize=11)
    ax.set_xlabel("Monat", fontsize=12)
    ax.set_ylabel("Alle Abschlüsse", fontsize=12)
    full_months = pd.date_range(start=full_y.index.min(), end=full_y.index.max(), freq="MS")
    ax.set_xticks(full_months)
    ax.set_xticklabels([d.strftime("%b %Y") for d in full_months], rotation=45, fontsize=9)
    ax.grid(True, linestyle='-', alpha=0.6)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_PATH, f"{title_suffix} - Forecast.png"), dpi=dpi, bbox_inches="tight")
    plt.show()

    # Einfluss-Plot
    X = df_monthly[features].dropna()
    y = df_monthly["Zielvariable_Gesamt"].loc[X.index]
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), index=X.index, columns=X.columns)
    model = LinearRegression(positive=True)
    model.fit(X_scaled, y)
    importance = pd.Series(model.coef_, index=features).sort_values()

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    importance.plot(kind="barh", color=blue_color, ax=ax)
    ax.set_title(f"Einfluss der Features – {title_suffix}", fontsize=13)
    ax.set_xlabel("Gewicht (standardisiert)", fontsize=12)
    ax.set_ylabel("Feature", fontsize=12)
    ax.grid(True, axis="x", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_PATH, f"{title_suffix} - Einfluss.png"), dpi=dpi, bbox_inches="tight")
    plt.show()

    # SHAP-Analyse
    explainer = shap.Explainer(model.predict, X_scaled)
    shap_values = explainer(X_scaled)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    shap.plots.beeswarm(
        shap_values,
        max_display=len(X_scaled.columns),
        color=plt.get_cmap("coolwarm"),
        show=False
    )
    plt.title(f"Einfluss der Features – {title_suffix} (SHAP)", fontsize=7)
    plt.xlabel("SHAP-Wert (Einfluss auf Prognose)", fontsize=6)
    plt.ylabel("Feature", fontsize=6)
    cbar = plt.gcf().axes[-1]  
    cbar.tick_params(labelsize=5) 
    cbar.set_ylabel("Feature value", fontsize=5,labelpad=15)
    plt.xticks(fontsize=5)
    plt.yticks(fontsize=5)
    plt.grid(True, axis="x", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_PATH, f"{title_suffix} - SHAP.png"), dpi=dpi, bbox_inches="tight")
    plt.show()

# Plots erzeugen
plot_forecast(dates_final, trues_final, preds_final, r2_final, mae_final, err_final, features_final, "Lineare Regression – Optuna-optimiert")


## Optuna optimiert - Lasso Regression mit Expanding Widnow

In [ ]:
# Feature-Lags
df_lagged = df_monthly.copy()

for f in stakeholder_features + optional_features:
    for lag in range(1, 4):
        df_lagged[f"{f} (Lag{lag})"] = df_lagged[f].shift(lag)

# Zielvariable
y_target = df_lagged["Zielvariable_Gesamt"]

# Forecast-Funktion
def run_forecast(X_raw, y, start_block=start_block):
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), index=X_raw.index, columns=X_raw.columns)
    trues, preds, dates = [], [], []
    for i in range(start_block, len(X_scaled)):
        X_train, y_train = X_scaled.iloc[:i], y.iloc[:i]
        X_test, y_test = X_scaled.iloc[i:i+1], y.iloc[i:i+1]
        model = LassoCV(cv=5, max_iter=500000,random_state=rs)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)[0]
        preds.append(y_pred)
        trues.append(y_test.values[0])
        dates.append(y_test.index[0])
    r2 = r2_score(trues, preds)
    mae = mean_absolute_error(trues, preds)
    err = mae / np.mean(trues) * 100
    return dates, trues, preds, r2, mae, err

# Early Stopping Callback
def early_stopping_callback(patience=patience_high):
    best_value = [float("inf")]
    counter = [0]
    def callback(study, trial):
        if trial.value < best_value[0]:
            best_value[0] = trial.value
            counter[0] = 0
        else:
            counter[0] += 1
        if counter[0] >= patience:
            raise optuna.exceptions.TrialPruned()
    return callback

# Optuna-Funktion
def optimize_features(y, start_block=start_block, n_trials=optuna_trials, patience=patience_high):
    study = optuna.create_study(direction="minimize")
    try:
        def objective(trial):
            selected = []
            for f in stakeholder_features:
                lagged_f = trial.suggest_categorical(f, [f"{f} (Lag1)", f"{f} (Lag2)", f"{f} (Lag3)"])
                selected.append(lagged_f)
            for f in optional_features:
                use_feature = trial.suggest_categorical(f + "_use", [True, False])
                if use_feature:
                    lagged_f = trial.suggest_categorical(f + "_lag", [f"{f} (Lag1)", f"{f} (Lag2)", f"{f} (Lag3)"])
                    selected.append(lagged_f)
            if trial.suggest_categorical("Arbeitstage_use", [True, False]):
                selected.append("Arbeitstage")
            if len(selected) == 0:
                return float("inf")
            if len(selected) > 8:
                selected = selected[:8]
            X = df_lagged[selected].dropna()
            y_clean = y.loc[X.index]
            _, _, _, _, mae, _ = run_forecast(X, y_clean, start_block)
            return mae
        study.optimize(objective, n_trials=n_trials, callbacks=[early_stopping_callback(patience)])
    except optuna.exceptions.TrialPruned:
        pass

    best_trial = study.best_trial
    selected = [best_trial.params[f] for f in stakeholder_features]
    for f in optional_features:
        if best_trial.params.get(f + "_use", False):
            selected.append(best_trial.params[f + "_lag"])
    if best_trial.params.get("Arbeitstage_use", False):
        selected.append("Arbeitstage")
    if len(selected) > 8:
        selected = selected[:8]
    return selected, study.best_value

# Lasso Regression – Optuna-optimiert
features_final, _ = optimize_features(y_target)
X_final = df_lagged[features_final].dropna()
dates_final, trues_final, preds_final, r2_final, mae_final, err_final = run_forecast(X_final, y_target.loc[X_final.index])

# Plot-Funktion
warnings.filterwarnings("ignore", category=ConvergenceWarning)
def plot_forecast(dates, trues, preds, r2, mae, err_rate, features, title_suffix):
    # Forecast-Plot
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    full_y = df_monthly["Zielvariable_Gesamt"]
    ax.plot(full_y.index, full_y, label="Reale Abschlüsse", color=grey_color, marker="o")
    ax.plot(dates, preds, label=f"{title_suffix}", color=blue_color, linestyle="--", marker="x")
    ax.axvline(x=dates[0], color=grey_color, linestyle="--", label="Beginn Prognose")
    ax.set_title(f"{title_suffix}\nR² = {r2:.3f}, MAE = {mae:.2f}, Fehlerquote = {err_rate:.2f}%", fontsize=13)
    features_string = ", ".join(features)
    plt.suptitle(f"Verwendete Features:\n{features_string}", y=1.05, fontsize=11)
    ax.set_xlabel("Monat", fontsize=12)
    ax.set_ylabel("Alle Abschlüsse", fontsize=12)
    full_months = pd.date_range(start=full_y.index.min(), end=full_y.index.max(), freq="MS")
    ax.set_xticks(full_months)
    ax.set_xticklabels([d.strftime("%b %Y") for d in full_months], rotation=45, fontsize=9)
    ax.grid(True, linestyle='-', alpha=0.6)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_PATH, f"Lasso_Forecast.png"), dpi=dpi, bbox_inches="tight")
    plt.show()

    # Einfluss-Plot

    X = df_lagged[features].dropna()
    y = df_lagged["Zielvariable_Gesamt"].loc[X.index]
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), index=X.index, columns=X.columns)
    model = LassoCV(cv=5, max_iter=optuna_trials)
    model.fit(X_scaled, y)
    importance = pd.Series(model.coef_, index=features).sort_values()

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    importance.plot(kind="barh", color=blue_color, ax=ax)
    ax.set_title(f"Einfluss der Features – {title_suffix}", fontsize=13)
    ax.set_xlabel("Gewicht (standardisiert)", fontsize=12)
    ax.set_ylabel("Feature", fontsize=12)
    ax.grid(True, axis="x", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_PATH, f"Lasso_Importance.png"), dpi=dpi, bbox_inches="tight")
    plt.show()

    # SHAP-Plot
  
    explainer = shap.Explainer(model, X_scaled)
    shap_values = explainer(X_scaled)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    shap.plots.beeswarm(
        shap_values,
        max_display=len(X_scaled.columns),
        color=plt.get_cmap("coolwarm"),
        show=False
    )
    plt.title(f"Einfluss der Features – {title_suffix} (SHAP)", fontsize=7)
    plt.xlabel("SHAP-Wert (Einfluss auf Vorhersage)", fontsize=6)
    plt.ylabel("Feature", fontsize=6)
    cbar = plt.gcf().axes[-1]
    cbar.tick_params(labelsize=5)
    cbar.set_ylabel("Feature value", fontsize=5, labelpad=15)
    plt.xticks(fontsize=5)
    plt.yticks(fontsize=5)
    plt.grid(True, axis="x", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_PATH, f"Lasso_SHAP.png"), dpi=dpi, bbox_inches="tight")
    plt.show()


# Plots erzeugen
plot_forecast(dates_final, trues_final, preds_final, r2_final, mae_final, err_final, features_final, "Lasso Regression – Optuna-optimiert")

## Optuna optimiert - SARIMAX mit Expanding Window

In [ ]:
# Feature-Lags
optional_features = ["Mietpreisindex", "Zielvariable Gesamt (Vormonat)"]
fixed_features = [f"{f} (Vormonat)" for f in stakeholder_features]
y = df_monthly["Zielvariable_Gesamt"]

# Expanding Forecast mit SARIMAX
def run_sarimax_forecast(X_raw, y, start_block, order, seasonal_order):
    trues, preds, dates = [], [], []
    for i in range(start_block, len(X_raw)):
        X_train, y_train = X_raw.iloc[:i], y.iloc[:i]
        X_test, y_test = X_raw.iloc[i:i+1], y.iloc[i:i+1]
        model = SARIMAX(endog=y_train,
                        exog=X_train,
                        order=order,
                        seasonal_order=seasonal_order,
                        enforce_stationarity=False,
                        enforce_invertibility=False)
        results = model.fit(disp=False)
        pred_series = results.forecast(steps=1, exog=X_test)
        pred = pred_series.iloc[0]           
        preds.append(pred)
        trues.append(y_test.values[0])
        dates.append(y_test.index[0])
    r2 = r2_score(trues, preds)
    mae = mean_absolute_error(trues, preds)
    err = mae / np.mean(trues) * 100
    return dates, trues, preds, r2, mae, err, results

# Optuna-Optimierung
def objective(trial):
    selected = [f for f in optional_features if trial.suggest_categorical(f, [True, False])] + fixed_features
    X = df_monthly[selected].dropna()
    y_clean = y.loc[X.index]

    order = (
        trial.suggest_int("p", 0, 2),
        trial.suggest_int("d", 0, 1),
        trial.suggest_int("q", 0, 2),
    )
    seasonal_order = (
        trial.suggest_int("P", 0, 2),
        trial.suggest_int("D", 0, 1),
        trial.suggest_int("Q", 0, 1),
        12
    )

    try:
        _, _, _, _, mae, _, _ = run_sarimax_forecast(X, y_clean, 12, order, seasonal_order)
    except:
        return float("inf")
    return mae

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=optuna_trials)

# Bestes Ergebnis
best_features = [f for f in optional_features if study.best_trial.params.get(f, False)] + fixed_features
order = (study.best_trial.params["p"], study.best_trial.params["d"], study.best_trial.params["q"])
seasonal_order = (study.best_trial.params["P"], study.best_trial.params["D"], study.best_trial.params["Q"], 12)

X_final = df_monthly[best_features].dropna()
y_final = y.loc[X_final.index]
dates_final, trues_final, preds_final, r2_final, mae_final, err_final, model_final = run_sarimax_forecast(
    X_final, y_final, 12, order, seasonal_order
)

# Forecast-Plot
fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
ax.plot(df_monthly["Zielvariable_Gesamt"], label="Reale Abschlüsse", color=grey_color, marker="o")
ax.plot(dates_final, preds_final, label="SARIMAX", color=blue_color, linestyle="--", marker="x")
ax.axvline(x=dates_final[0], color=grey_color, linestyle="--", label="Beginn Prognose")
ax.set_title(f"SARIMAX\nR² = {r2_final:.3f}, MAE = {mae_final:.2f}, Fehlerquote = {err_final:.2f}%", fontsize=13)
features_str = ", ".join(best_features)
plt.suptitle(f"Verwendete Features:\n{features_str}", y=1.05, fontsize=11)
ax.set_xlabel("Monat", fontsize=12)
ax.set_ylabel("Alle Abschlüsse", fontsize=12)
ax.set_xticks(pd.date_range(start=df_monthly.index.min(), end=df_monthly.index.max(), freq="MS"))
ax.set_xticklabels([d.strftime("%b %Y") for d in pd.date_range(start=df_monthly.index.min(), end=df_monthly.index.max(), freq="MS")], rotation=45, fontsize=9)
ax.grid(True, linestyle='-', alpha=0.6)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "SARIMAX_Forecast.png"), dpi=dpi, bbox_inches="tight")
plt.show()

# Einfluss der Features 
num_exog = X_final.shape[1]
exog_params = model_final.params[-num_exog:]  
exog_params.index = X_final.columns 

scaler = StandardScaler().fit(X_final)
X_scaled = pd.DataFrame(scaler.transform(X_final), index=X_final.index, columns=X_final.columns)

coefs = pd.Series(exog_params.values, index=X_scaled.columns)

fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
coefs.sort_values().plot(kind="barh", color=blue_color, ax=ax)
ax.set_title("Einfluss der Features – SARIMAX (standardisiert)", fontsize=13)
ax.set_xlabel("Gewicht (Koeffizient)", fontsize=12)
ax.set_ylabel("Feature", fontsize=12)
plt.grid(True, axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "SARIMAX_Importance.png"), dpi=dpi, bbox_inches="tight")
plt.show()

# SHAP-ähnlicher Plot
impact = coefs * X_scaled.std()

fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
impact.sort_values().plot(kind="barh", color=blue_color, ax=ax)
ax.set_title("Einfluss der Features – SARIMAX (SHAP-ähnlich)", fontsize=13)
ax.set_xlabel("Einfluss (Gewicht × Standardabweichung)", fontsize=12)
ax.set_ylabel("Feature", fontsize=12)
plt.grid(True, axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "SARIMAX_SHAP.png"), dpi=dpi, bbox_inches="tight")
plt.show()


## Optuana optmiert - XGBoost mit Expanding Window

In [ ]:
# Feature-Lags
for f in stakeholder_features:
    df_monthly[f"{f} (Vormonat)"] = df_monthly[f].shift(1)

# Feature-Sets
stakeholder_features = [f"{f} (Vormonat)" for f in stakeholder_features]
optional_features = [
    "Transaktionspreisindex Einfamilienhäuser",
    "Transaktionspreisindex Eigentumswohnung",
    "Arbeitstage",
    "Mietpreisindex",
    "Zielvariable Gesamt (Vormonat)"
]
y_target = df_monthly["Zielvariable_Gesamt"]

# Forecast-Funktion
def run_forecast(X_raw, y, start_block=start_block, model_class=XGBRegressor, model_kwargs=None):
    if model_kwargs is None:
        model_kwargs = {}
    trues, preds, dates = [], [], []
    for i in range(start_block, len(X_raw)):
        X_train, y_train = X_raw.iloc[:i], y.iloc[:i]
        X_test, y_test = X_raw.iloc[i:i+1], y.iloc[i:i+1]
        model = model_class(**model_kwargs)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)[0]
        preds.append(y_pred)
        trues.append(y_test.values[0])
        dates.append(y_test.index[0])
    r2 = r2_score(trues, preds)
    mae = mean_absolute_error(trues, preds)
    err = mae / np.mean(trues) * 100
    return dates, trues, preds, r2, mae, err, model

# Gemeinsame Feature- & Hyperparameter-Optimierung
def optimize_features_and_hyperparams(fixed_features, optional_features, y, start_block=start_block, n_trials=10):
    def objective(trial):
        selected = [f for f in optional_features if trial.suggest_categorical(f, [True, False])]
        selected += fixed_features
        if not selected:
            return float("inf")

        X = df_monthly[selected].dropna()
        y_clean = y.loc[X.index]

        params = {
            "max_depth": trial.suggest_int("max_depth", 2, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "gamma": trial.suggest_float("gamma", 0, 5),
            "reg_alpha": trial.suggest_float("reg_alpha", 0, 10),
            "reg_lambda": trial.suggest_float("reg_lambda", 0, 10),
            "verbosity": 0
        }

        try:
            _, _, _, _, mae, _, _ = run_forecast(X, y_clean, start_block, model_class=XGBRegressor, model_kwargs=params)
        except:
            return float("inf")
        return mae

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    best_features = [f for f in optional_features if study.best_trial.params.get(f, False)] + fixed_features
    best_params = {k: v for k, v in study.best_trial.params.items() if k not in optional_features}
    return best_features, best_params, study.best_value

# Optimierung starten
features_final, best_params, _ = optimize_features_and_hyperparams(
    stakeholder_features,
    optional_features,
    y_target,
    start_block=start_block,
    n_trials=optuna_trials
)

# Forecast mit besten Parametern
X_final = df_monthly[features_final].dropna()
y_final = y_target.loc[X_final.index]
dates_final, trues_final, preds_final, r2_final, mae_final, err_final, model_final = run_forecast(
    X_final, y_final, start_block=start_block, model_class=XGBRegressor, model_kwargs=best_params
)

# Forecast-Plot
fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
ax.plot(df_monthly["Zielvariable_Gesamt"], label="Reale Abschlüsse", color=grey_color, marker="o")
ax.plot(dates_final, preds_final, label="XGBoost", color=blue_color, linestyle="--", marker="x")
ax.axvline(x=dates_final[0], color=grey_color, linestyle="--", label="Beginn Prognose")
ax.set_title(f"XGBoost\nR² = {r2_final:.3f}, MAE = {mae_final:.2f}, Fehlerquote = {err_final:.2f}%", fontsize=13)
features_str = ", ".join(features_final)
plt.suptitle(f"Verwendete Features:\n{features_str}", y=1.05, fontsize=11)
ax.set_xlabel("Monat", fontsize=12)
ax.set_ylabel("Alle Abschlüsse", fontsize=12)
ax.set_xticks(pd.date_range(start=df_monthly.index.min(), end=df_monthly.index.max(), freq="MS"))
ax.set_xticklabels([d.strftime("%b %Y") for d in pd.date_range(start=df_monthly.index.min(), end=df_monthly.index.max(), freq="MS")], rotation=45, fontsize=9)
ax.grid(True, linestyle='-', alpha=0.6)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "XGBoost_Forecast.png"), dpi=dpi, bbox_inches="tight")
plt.show()

# Gain-basierte Feature Importance
importances = pd.Series(model_final.feature_importances_, index=X_final.columns).sort_values()
fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
importances.plot(kind="barh", color=blue_color, ax=ax)
ax.set_title("Einfluss der Features – XGBoost (Gain-basiert)", fontsize=13)
ax.set_xlabel("Wichtigkeit (Gain)", fontsize=12)
ax.set_ylabel("Feature", fontsize=12)
ax.tick_params(axis='x', labelsize=9)
ax.tick_params(axis='y', labelsize=9)
ax.grid(True, axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "XGBoost_Importance.png"), dpi=dpi, bbox_inches="tight")
plt.show()

# SHAP-Analyse
explainer = shap.Explainer(model_final)
shap_values = explainer(X_final)
fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
shap.plots.beeswarm(
    shap_values,
    max_display=len(X_final.columns),
    color=plt.get_cmap("coolwarm"),
    show=False
)
plt.title("Einfluss der Features – XGBoost (SHAP)", fontsize=7)
plt.suptitle("Standardisierte SHAP-Werte (Einfluss auf Prognose)", y=1.02, fontsize=6)
plt.xlabel("SHAP-Wert (Einfluss auf Vorhersage)", fontsize=6)
plt.ylabel("Feature", fontsize=6)
cbar = plt.gcf().axes[-1]
cbar.tick_params(labelsize=5)
cbar.set_ylabel("Feature value", fontsize=5, labelpad=15)
plt.xticks(fontsize=5)
plt.yticks(fontsize=5)
plt.grid(True, axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(FIG_PATH, "XGBoost_SHAP.png"), dpi=dpi, bbox_inches="tight")
plt.show()


## Optuna optimiert - Ridge Regression mit Expandign Window

In [ ]:
# Feature-Lags
df_lagged = df_monthly.copy()

# Alle Lags generieren
for f in stakeholder_features + optional_features:
    for lag in range(1, 4):
        df_lagged[f"{f} (Lag{lag})"] = df_lagged[f].shift(lag)

# Zielvariable
y_target = df_lagged["Zielvariable_Gesamt"]

# Forecast-Funktion
def run_forecast(X_raw, y, start_block=start_block):
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), index=X_raw.index, columns=X_raw.columns)
    trues, preds, dates = [], [], []
    for i in range(start_block, len(X_scaled)):
        X_train, y_train = X_scaled.iloc[:i], y.iloc[:i]
        X_test, y_test = X_scaled.iloc[i:i+1], y.iloc[i:i+1]
        model = RidgeCV(alphas=np.logspace(-4, 4, 100), cv=5)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)[0]
        preds.append(y_pred)
        trues.append(y_test.values[0])
        dates.append(y_test.index[0])
    r2 = r2_score(trues, preds)
    mae = mean_absolute_error(trues, preds)
    err = mae / np.mean(trues) * 100
    return dates, trues, preds, r2, mae, err

# Early Stopping Callback
def early_stopping_callback(patience=patience_low):
    best_value = [float("inf")]
    counter = [0]
    def callback(study, trial):
        if trial.value < best_value[0]:
            best_value[0] = trial.value
            counter[0] = 0
        else:
            counter[0] += 1
        if counter[0] >= patience:
            raise optuna.exceptions.TrialPruned()
    return callback

# Optuna-Funktion
def optimize_features(y, start_block=start_block, n_trials=optuna_trials, patience=patience_low):
    study = optuna.create_study(direction="minimize")
    try:
        def objective(trial):
            selected = []

            # Stakeholder: genau ein Lag pro Feature
            for f in stakeholder_features:
                lagged_f = trial.suggest_categorical(f, [f"{f} (Lag1)", f"{f} (Lag2)", f"{f} (Lag3)"])
                selected.append(lagged_f)

            # Optional: höchstens ein Lag pro Feature
            for f in optional_features:
                use_feature = trial.suggest_categorical(f + "_use", [True, False])
                if use_feature:
                    lagged_f = trial.suggest_categorical(f + "_lag", [f"{f} (Lag1)", f"{f} (Lag2)", f"{f} (Lag3)"])
                    selected.append(lagged_f)

            # Optional: Arbeitstage ohne Lag
            if trial.suggest_categorical("Arbeitstage_use", [True, False]):
                selected.append("Arbeitstage")

            # Begrenzung auf maximal 8 Features
            if len(selected) == 0:
                return float("inf")
            if len(selected) > 8:
                selected = selected[:8]

            X = df_lagged[selected].dropna()
            y_clean = y.loc[X.index]
            _, _, _, _, mae, _ = run_forecast(X, y_clean, start_block)
            return mae

        study.optimize(objective, n_trials=n_trials, callbacks=[early_stopping_callback(patience)])
    except optuna.exceptions.TrialPruned:
        pass

    best_trial = study.best_trial
    selected = []

    for f in stakeholder_features:
        selected.append(best_trial.params[f])
    for f in optional_features:
        if best_trial.params.get(f + "_use", False):
            selected.append(best_trial.params[f + "_lag"])
    if best_trial.params.get("Arbeitstage_use", False):
        selected.append("Arbeitstage")

    if len(selected) > 8:
        selected = selected[:8]

    return selected, study.best_value

# Ridge Regression – Optuna-optimiert
features_final, _ = optimize_features(y_target)
X_final = df_lagged[features_final].dropna()
dates_final, trues_final, preds_final, r2_final, mae_final, err_final = run_forecast(X_final, y_target.loc[X_final.index])


# Plot-Funktion
def plot_forecast(dates, trues, preds, r2, mae, err_rate, features, title_suffix):
    features_string = ", ".join(features)

    # Forecast-Plot
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    full_y = df_monthly["Zielvariable_Gesamt"]
    ax.plot(full_y.index, full_y, label="Reale Abschlüsse", color=grey_color, marker="o")
    ax.plot(dates, preds, label=title_suffix, color=blue_color, linestyle="--", marker="x")
    ax.axvline(x=dates[0], color=grey_color, linestyle="--", label="Beginn Prognose")
    ax.set_title(f"{title_suffix}\nR² = {r2:.3f}, MAE = {mae:.2f}, Fehlerquote = {err_rate:.2f}%", fontsize=13)
    plt.suptitle(f"Verwendete Features:\n{features_string}", y=1.05, fontsize=11)
    ax.set_xlabel("Monat", fontsize=12)
    ax.set_ylabel("Alle Abschlüsse", fontsize=12)
    full_months = pd.date_range(start=full_y.index.min(), end=full_y.index.max(), freq="MS")
    ax.set_xticks(full_months)
    ax.set_xticklabels([d.strftime("%b %Y") for d in full_months], rotation=45, fontsize=9)
    ax.grid(True, linestyle='-', alpha=0.6)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_PATH, f"{title_suffix}_Forecast.png"), dpi=dpi, bbox_inches="tight")
    plt.show()

    # Einfluss-Plot (barh)
    X = df_lagged[features].dropna()
    y = df_lagged["Zielvariable_Gesamt"].loc[X.index]

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), index=X.index, columns=X.columns)
    model = RidgeCV(alphas=np.logspace(-4, 4, 100), cv=5)
    model.fit(X_scaled, y)
    importance = pd.Series(model.coef_, index=features).sort_values()

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    importance.plot(kind="barh", color=blue_color, ax=ax)
    ax.set_title(f"Einfluss der Features – {title_suffix}", fontsize=13)
    ax.set_xlabel("Gewicht (standardisiert)", fontsize=12)
    ax.set_ylabel("Feature", fontsize=12)
    ax.grid(True, axis="x", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_PATH, f"{title_suffix}_Einfluss.png"), dpi=dpi, bbox_inches="tight")
    plt.show()

    # SHAP-Beeswarm-Plot
    explainer = shap.Explainer(model, X_scaled)

    shap_values = explainer(X_scaled)

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    shap.plots.beeswarm(shap_values, max_display=len(X_scaled.columns), color=plt.get_cmap("coolwarm"), show=False)
    plt.title(f"Einfluss der Features – {title_suffix} (SHAP)", fontsize=7)
    plt.xlabel("SHAP-Wert (Einfluss auf Vorhersage)", fontsize=6)
    plt.ylabel("Feature", fontsize=6)
    cbar = plt.gcf().axes[-1] 
    cbar.tick_params(labelsize=5)  
    cbar.set_ylabel("Feature value", fontsize=5,labelpad=15)
    plt.xticks(fontsize=5)
    plt.yticks(fontsize=5)
    plt.grid(True, axis="x", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_PATH, f"{title_suffix}_SHAP.png"), dpi=dpi, bbox_inches="tight")
    plt.show()

# Plot erzeugen
plot_forecast(dates_final, trues_final, preds_final, r2_final, mae_final, err_final, features_final, "Ridge Regression – Optuna-optimiert")